In [65]:
from datetime import datetime, timedelta, date
import pandas as pd
import requests
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from SmartApi import SmartConnect
from pyotp import TOTP
import requests
from SmartApi.smartWebSocketV2 import SmartWebSocketV2
import time
from dotenv import load_dotenv
import os
import threading

### India VIX calculation

In [ ]:
start_date = (datetime.today() - timedelta(days=365)).date()
df = yf.download("^INDIAVIX", start=start_date)

iv = np.array(df['Close'])
temp = []
for i in iv:
    temp.append(i[0])
iv = np.array(temp)
del temp, i
current_iv = iv[-1]       
iv_min = np.min(iv)             
iv_max = np.max(iv)             

# IV Rank = (Current IV - Min IV) / (Max IV - Min IV) * 100
iv_rank = (current_iv - iv_min) / (iv_max - iv_min) * 100

# IV Percentile = % of days IV was below current IV
iv_percentile = np.sum(iv <= current_iv) / len(iv) * 100

print(f"Today's IV: {current_iv:.2f}")
print(f"IV Rank: {iv_rank:.2f}%")
print(f"IV Percentile: {iv_percentile:.2f}%")

### EDA

In [ ]:
start_date = (datetime.today() - timedelta(days=365)).date()
end_date = datetime.today().date()
nifty = yf.download("^NSEI", start=start_date, end=end_date)['Close']
vix = yf.download("^INDIAVIX", start=start_date, end=end_date)['Close']
fig, ax1 = plt.subplots(figsize=(12,6))

color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('NIFTY50', color=color)
ax1.plot(nifty.index, nifty.values, color=color, label='NIFTY50')
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx()
plt.grid(False)
color = 'tab:red'
ax2.set_ylabel('IV', color=color)
color = 'tab:red'
ax2.set_ylabel('India VIX', color=color)
ax2.plot(vix.index, vix.values, color=color, label='India VIX')
ax2.tick_params(axis='y', labelcolor=color)
fig.suptitle("NIFTY50 vs India VIX (Last 1 Year)")
fig.tight_layout()
plt.show();

In [ ]:
combined = pd.concat([nifty, vix], axis=1)
combined.columns = ['NIFTY50', 'INDIAVIX']  
sns.heatmap(combined.corr(), annot=True);

### Websocket

In [ ]:
yf.download("^NSEI")['Close'].iloc[-1]

In [47]:
url = 'https://margincalculator.angelbroking.com/OpenAPI_File/files/OpenAPIScripMaster.json'
d = requests.get(url).json()
token_df = pd.DataFrame(d)
token_df['expiry'] = pd.to_datetime(token_df['expiry'], format='mixed').dt.date
token_df['strike'] = pd.to_numeric(token_df['strike'], errors='coerce')
nifty_calls = token_df[(token_df['name'] == 'NIFTY') & (token_df['instrumenttype'] == 'OPTIDX')].copy()

# Separate CE and PE, sort by expiry and strike
nifty_calls['option_type'] = nifty_calls['symbol'].str[-2:] 
nifty_calls = nifty_calls.sort_values(['expiry', 'strike']).reset_index(drop=True)
nifty_calls = nifty_calls[nifty_calls['option_type']=='CE']
nifty = nifty_calls[['token', 'symbol', 'name', 'expiry', 'strike']]
nifty = nifty.copy()
nifty['expiry'] = pd.to_datetime(nifty['expiry'])
nifty['strike'] = nifty['strike'] / 100

In [48]:
nifty.head(10)

,token,symbol,name,expiry,strike
1,62914,NIFTY30MAR2619000CE,NIFTY,2026-03-30,19000.0
2,62916,NIFTY30MAR2620000CE,NIFTY,2026-03-30,20000.0
4,45610,NIFTY30MAR2620500CE,NIFTY,2026-03-30,20500.0
6,45612,NIFTY30MAR2620550CE,NIFTY,2026-03-30,20550.0
8,45616,NIFTY30MAR2620600CE,NIFTY,2026-03-30,20600.0
11,44464,NIFTY30MAR2620650CE,NIFTY,2026-03-30,20650.0
12,54409,NIFTY30MAR2620700CE,NIFTY,2026-03-30,20700.0
14,54411,NIFTY30MAR2620750CE,NIFTY,2026-03-30,20750.0
16,54413,NIFTY30MAR2620800CE,NIFTY,2026-03-30,20800.0
18,54415,NIFTY30MAR2620850CE,NIFTY,2026-03-30,20850.0


In [61]:
spot = 22331.4
today = pd.to_datetime('today').normalize()

long_calls = nifty[(nifty['strike'] >= spot*0.60) & (nifty['strike'] <= spot) &
    (nifty['expiry'] >= today + pd.Timedelta(days=90)) & (nifty['expiry'] <= today + pd.Timedelta(days=150))].copy()

short_calls = nifty[(nifty['strike'] >= spot) & (nifty['strike'] <= spot*1.3) &
    (nifty['expiry'] >= today + pd.Timedelta(days=10)) & (nifty['expiry'] <= today + pd.Timedelta(days=30))].copy()

In [62]:
long_calls

,token,symbol,name,expiry,strike
1750,50447,NIFTY30JUN2615000CE,NIFTY,2026-06-30,15000.0
1753,55181,NIFTY30JUN2616500CE,NIFTY,2026-06-30,16500.0
1755,50556,NIFTY30JUN2618000CE,NIFTY,2026-06-30,18000.0
1758,39989,NIFTY30JUN2619000CE,NIFTY,2026-06-30,19000.0
1759,55183,NIFTY30JUN2619500CE,NIFTY,2026-06-30,19500.0
1762,51453,NIFTY30JUN2621000CE,NIFTY,2026-06-30,21000.0


In [63]:
short_calls

,token,symbol,name,expiry,strike
636,54695,NIFTY13APR2622350CE,NIFTY,2026-04-13,22350.0
639,54697,NIFTY13APR2622400CE,NIFTY,2026-04-13,22400.0
640,54699,NIFTY13APR2622450CE,NIFTY,2026-04-13,22450.0
642,54701,NIFTY13APR2622500CE,NIFTY,2026-04-13,22500.0
645,54703,NIFTY13APR2622550CE,NIFTY,2026-04-13,22550.0
...,...,...,...,...,...
1269,72756,NIFTY28APR2627500CE,NIFTY,2026-04-28,27500.0
1270,72760,NIFTY28APR2627550CE,NIFTY,2026-04-28,27550.0
1272,72762,NIFTY28APR2627600CE,NIFTY,2026-04-28,27600.0
1274,72764,NIFTY28APR2627650CE,NIFTY,2026-04-28,27650.0


In [66]:
# Creds
load_dotenv()
api_key   = os.getenv("API_KEY")
client_code = os.getenv("CLIENT_CODE")
pwd  = os.getenv("PWD")
totp  = os.getenv("TOTP_KEY")
# Connection
obj = SmartConnect(api_key=api_key)
session = obj.generateSession(client_code, pwd, TOTP(totp).now())
refreshToken = session['data']['refreshToken']
authToken = session['data']['jwtToken']
feedToken = obj.getfeedToken()
res = obj.getProfile(refreshToken)
obj.generateToken(refreshToken)
res=res['data']['exchanges']

[I 260330 16:49:49 smartConnect:124] in pool


In [33]:
spot = obj.ltpData(exchange="NSE", tradingsymbol="NIFTY", symboltoken="99926000")['data']['ltp']
spot

22331.4

In [ ]:
# ---------WebSocketV2---------
sws = SmartWebSocketV2(authToken, api_key, client_code, feedToken, max_retry_attempt=5)
correlation_id = "abcd"
action = 1
mode = 1
token_list = [{"exchangeType": 2, "tokens": ["54490"]}]

def on_open(wsapp):
    print("on open")
    sws.subscribe(correlation_id, mode, token_list)

def on_data(wsapp, message):
    print("Ticks: {}".format(message))

def on_error(wsapp, error):
    print(error)

def on_close(wsapp):
    print("Close")

def close_connection():
    sws.close_connection()

sws.on_open = on_open
sws.on_data = on_data
sws.on_error = on_error
sws.on_close = on_close

# sws.connect()

In [ ]:
# Put Call Ratio
obj.putCallRatio()

# Greeks
params = {"name": "NIFTY", "expirydate": "30JUN2026"}
temp = obj.optionGreek(params)
greeks_df = pd.DataFrame(temp['data'])
greeks_df